<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">


# Python for Finance, 3rd Edition
## Chapter 16 · NLP and LLM Foundations
&copy; Dr. Yves J. Hilpisch<br>
AI-supported by various LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
This notebook mirrors the current chapter-16 draft with executable examples
for tokenization, sparse text vectors, a baseline classifier, and a toy
self-attention calculation.


### How to Use This Notebook
- Run the cells from top to bottom the first time so later sections can
  reuse earlier variables.
- Keep placeholders for model identifiers and API keys out of version
  control (use environment variables).
- Use the chapter text for the surrounding interpretation and caveats.


In [ ]:
from pathlib import Path
import subprocess
import sys

NOTEBOOK_SUBDIR = "notebooks"
COLAB_PACKAGES = {"openai": "openai"}
REPO_NAME = "py4fi3rd"
REPO_URL = "https://github.com/yhilpisch/py4fi3rd.git"


def _support_dir() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        support_dir = candidate / "notebooks"
        if (support_dir / "_book_notebook_support.py").exists():
            return support_dir
    if "google.colab" in sys.modules:
        root = Path("/content") / REPO_NAME
        if not root.exists():
            subprocess.run(
                ["git", "clone", "--depth", "1", REPO_URL, str(root)],
                check=True,
            )
        return root / "notebooks"
    raise RuntimeError("Could not locate notebook support helpers.")


SUPPORT_DIR = _support_dir()
if str(SUPPORT_DIR) not in sys.path:
    sys.path.insert(0, str(SUPPORT_DIR))

from _book_notebook_support import setup_notebook

CONTEXT = setup_notebook(
    notebook_subdir=NOTEBOOK_SUBDIR,
    colab_packages=COLAB_PACKAGES,
)

PROJECT_ROOT = CONTEXT["PROJECT_ROOT"]
NOTEBOOK_DIR = CONTEXT["NOTEBOOK_DIR"]
CODE_DIR = CONTEXT["CODE_DIR"]
CHAPTERS_DIR = CONTEXT["CHAPTERS_DIR"]
FIGURES_DIR = CONTEXT["FIGURES_DIR"]
DATA_DIR = CONTEXT["DATA_DIR"]

PROJECT_ROOT

The focus here is on minimal, runnable building blocks: text to tokens,
tokens to vectors, vectors to baseline models, and a concrete
self-attention toy example.


## Text Data in Quantitative Finance
Use these examples as stand-ins for real corpora such as news headlines,
filings, transcripts, and internal logs.


## From Raw Text to Tokens
Start from raw strings and convert them into token sequences that can be
mapped to integers.


### Cleaning and Simple Tokenization
A minimal tokenizer lower-cases, strips non-alphanumerics, and splits on
whitespace.


In [ ]:
import re
import numpy as np

headlines = [
    "AAPL beats earnings expectations, shares jump",
    "Bank stocks slide as rates outlook shifts",
    "Energy sector rallies on higher oil prices",
    "Tech shares retreat after strong year-to-date gains",
]


def simple_tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\\s]", " ", text)
    tokens = text.split()
    return tokens


[simple_tokenize(h) for h in headlines]

### Building a Vocabulary
A vocabulary is a stable mapping from distinct tokens to integer IDs.


In [ ]:
all_tokens = [tok for h in headlines for tok in simple_tokenize(h)]

vocab = sorted(set(all_tokens))

token_to_id = {tok: i for i, tok in enumerate(vocab)}

vocab[:5], len(vocab)

### Encoding Headlines as Token IDs
With a vocabulary in place, you can turn token sequences into integer
sequences.


In [ ]:
def encode(tokens, mapping):
    return [mapping[t] for t in tokens]


encoded = [encode(simple_tokenize(h), token_to_id) for h in headlines]

encoded

## Numerical Representations of Text
Convert tokenized text into vectors that can be fed into classical ML models.


### Bag-of-Words and TF–IDF
Start with sparse document-term representations via `scikit-learn`
vectorizers.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(lowercase=True)

bow = vectorizer.fit_transform(headlines)

bow.toarray()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(lowercase=True)

X_tfidf = tfidf.fit_transform(headlines)

X_tfidf.shape

## Classical Text Models with scikit-learn
Use TF–IDF features as inputs to a baseline linear classifier.


In [ ]:
labels = np.array([1, -1, 1, -1])

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    labels,
    test_size=0.5,
    random_state=2027,
    stratify=labels,
)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred))

## Transformers and Self-Attention
A toy, explicit attention computation makes the matrix operations concrete.


### A Toy Self-Attention Calculation
Compute single-head self-attention weights step by step on a small token
sequence.


In [ ]:
import numpy as np

tokens = ["rates", "cut", "to", "boost", "equities"]

d_model = 4
rng = np.random.default_rng(2027)
X = rng.normal(scale=0.5, size=(len(tokens), d_model))

W_q = rng.normal(scale=0.5, size=(d_model, d_model))
W_k = rng.normal(scale=0.5, size=(d_model, d_model))
W_v = rng.normal(scale=0.5, size=(d_model, d_model))

Q = X @ W_q
K = X @ W_k
V = X @ W_v

scores = Q @ K.T / np.sqrt(d_model)


def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)


attn = softmax(scores, axis=-1)

attn.round(2)

### Combining Values into Contextualised Representations
Use the attention weights to aggregate value vectors into new token
representations.


In [ ]:
Z = attn @ V

Z.shape

### Visualising Attention Weights
This cell mirrors `code/figures/ch16_self_attention_heatmap.py` and
(re)generates the PNG under `assets/figures/` (requires `matplotlib`).


In [ ]:
from pathlib import Path

import numpy as np

try:
    import matplotlib as mpl
    import matplotlib.pyplot as plt
except ImportError:
    mpl = None
    plt = None


def _compute_attention(seed=2027):
    tokens = ["rates", "cut", "to", "boost", "equities"]
    d_model = 4
    rng = np.random.default_rng(seed)

    x = rng.normal(scale=0.5, size=(len(tokens), d_model))
    w_q = rng.normal(scale=0.5, size=(d_model, d_model))
    w_k = rng.normal(scale=0.5, size=(d_model, d_model))

    q = x @ w_q
    k = x @ w_k

    scores = q @ k.T / np.sqrt(d_model)
    scores = scores - scores.max(axis=-1, keepdims=True)
    e = np.exp(scores)
    attn = e / e.sum(axis=-1, keepdims=True)
    return attn, tokens


if mpl is None or plt is None:
    print("Skipping: matplotlib is not installed in this environment.")
else:
    mpl.style.use("seaborn-v0_8")
    mpl.rcParams.update(
        {
            "font.family": "serif",
            "font.size": 7,
            "figure.dpi": 300,
        }
    )

    attn_fig, tokens_fig = _compute_attention()
    fig, ax = plt.subplots(figsize=(3.4, 2.6))

    im = ax.imshow(
        attn_fig,
        cmap="viridis",
        vmin=0.0,
        vmax=float(attn_fig.max()),
    )

    ax.set_xticks(range(len(tokens_fig)))
    ax.set_yticks(range(len(tokens_fig)))
    ax.set_xticklabels(tokens_fig, rotation=30, ha="right", fontsize=7)
    ax.set_yticklabels(tokens_fig, fontsize=7)
    ax.set_xlabel("Key / value tokens", fontsize=7)
    ax.set_ylabel("Query tokens", fontsize=7)
    ax.set_title("Toy self-attention weights", fontsize=9)
    ax.tick_params(axis="both", labelsize=7)

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Attention weight", fontsize=7)
    cbar.ax.tick_params(labelsize=7)

    figures_dir = Path("..") / "assets" / "figures"
    figures_dir.mkdir(parents=True, exist_ok=True)
    outfile = figures_dir / "ch16_self_attention_heatmap.png"
    fig.tight_layout()
    fig.savefig(outfile, dpi=300)
    plt.close(fig)
    outfile

In [ ]:
from pathlib import Path
from IPython.display import Image, display

path = Path("..") / "assets" / "figures" / "ch16_self_attention_heatmap.png"
if path.exists():
    display(Image(str(path)))
else:
    print(
        f"Figure not found at {path!s}. "
        "Run the previous cell first."
    )


The goal here is to highlight stable call structure; model IDs and CLI
flags evolve, so the examples use placeholders.


### Example: Summarizing a Transcript via CLI

```bash
export MODEL_ID="..."
export OPENAI_API_KEY="..."
cat earnings_call.txt \
  | openai chat.completions.create \
      -m "$MODEL_ID" \
      -g "You are a concise finance analyst." \
      --input-file - \
      --temperature 0.2
```


### Example: Using an LLM from Python
Live API access is explicitly opt-in so validation never makes a network call
merely because credentials are present in the environment.


In [ ]:
import os

try:
    from openai import OpenAI
except ImportError:
    OpenAI = None


MODEL_ID = os.getenv("OPENAI_MODEL")
RUN_LIVE_API = os.getenv("RUN_OPENAI_EXAMPLE") == "1"

headline = "Bank shares slide after cautious guidance"

messages = [
    {
        "role": "system",
        "content": (
            "You are a sentiment classifier for finance headlines. "
            "Reply with one of: positive, neutral, negative."
        ),
    },
    {"role": "user", "content": headline},
]

if not RUN_LIVE_API:
    print(
        "Skipping live API call. Set RUN_OPENAI_EXAMPLE=1, "
        "OPENAI_MODEL, and OPENAI_API_KEY to run it."
    )
elif OpenAI is None:
    print("Skipping: install the 'openai' package to run this example.")
elif os.getenv("OPENAI_API_KEY") is None:
    print("Skipping: set OPENAI_API_KEY to run this example.")
elif MODEL_ID is None:
    print("Skipping: set OPENAI_MODEL to a model available to you.")
else:
    client = OpenAI()
    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=messages,
        temperature=0.0,
    )
    sentiment = response.choices[0].message.content.strip()
    sentiment

The RAG pseudo-code uses placeholders; the next cell defines minimal
stand-ins so you can execute the control flow locally.


In [ ]:
import hashlib
import numpy as np


def split_into_chunks(text, chunk_size=120):
    text = text.strip()
    if not text:
        return []
    return [text[i : i + chunk_size] for i in range(0, len(text), chunk_size)]


def embed_text(text, dim=32):
    seed = int.from_bytes(
        hashlib.sha256(text.encode("utf-8")).digest()[:8],
        "little",
    )
    rng = np.random.default_rng(seed)
    return rng.normal(size=dim)


class VectorStore:
    def __init__(self):
        self._items = []  # list of (embedding, metadata)

    def add(self, embedding, metadata):
        self._items.append((np.asarray(embedding, dtype=float), dict(metadata)))

    def search(self, query_embedding, top_k=5):
        q = np.asarray(query_embedding, dtype=float)
        qn = float(np.linalg.norm(q) + 1e-12)
        scored = []
        for emb, meta in self._items:
            score = float(np.dot(q, emb) / (qn * (np.linalg.norm(emb) + 1e-12)))
            meta2 = dict(meta)
            meta2["score"] = score
            scored.append((score, meta2))
        scored.sort(key=lambda x: x[0], reverse=True)
        return [m for _, m in scored[:top_k]]

In [ ]:
def build_index(documents):
    chunks = []  # list of {"id": ..., "text": ...}
    for doc_id, text in documents:
        for i, chunk in enumerate(split_into_chunks(text)):
            chunks.append({"id": (doc_id, i), "text": chunk})
    embeddings = [embed_text(c["text"]) for c in chunks]
    index = VectorStore()
    for emb, chunk in zip(embeddings, chunks):
        index.add(embedding=emb, metadata=chunk)
    return index


def answer_question(question, index, chat_model, k=5):
    q_emb = embed_text(question)
    neighbours = index.search(q_emb, top_k=k)
    context = "\n\n".join(n["text"] for n in neighbours)
    messages = [
        {
            "role": "system",
            "content": (
                "Answer the user's question using only the context "
                "provided below. If the context is insufficient, say so."
            ),
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {question}",
        },
    ]
    response = chat_model(messages=messages)
    return response["content"]

In [ ]:
# Example usage with placeholder components
documents = [
    ("f1", "First illustrative filing text ..."),
    ("f2", "Second illustrative research note ..."),
]
index = build_index(documents)


def chat_model(messages):
    """Placeholder stand-in for a real chat-completions call."""

    return {"content": "[model reply goes here]"}


answer = answer_question(
    question="What risks are mentioned across these documents?",
    index=index,
    chat_model=chat_model,
    k=3,
)
print(answer)

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">
